# Orchestrator-Subagent — Google ADK

**Pattern:** Orchestrator delegates to sub-agents via transfer

```
trip_orchestrator
  ├── highlights_specialist (tools: get_highlights)
  └── logistics_specialist  (tools: get_logistics)
```

Orchestrator transfers to each sub-agent in turn, then synthesizes the final package.

In [ ]:
import asyncio, os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
print("✓ Ready")

In [ ]:
def get_highlights(city: str) -> dict:
    """Get top tourist highlights.
    Args:
        city: City name.
    Returns:
        Dict with city and highlights list.
    """
    data = {"tokyo": ["Shibuya","Senso-ji","Tsukiji","Mt Fuji"], "paris": ["Eiffel Tower","Louvre","Notre Dame"], "bangalore": ["Lalbagh","Nandi Hills"]}
    return {"city": city, "highlights": data.get(city.lower(), [])} if city.lower() in data else {"error": f"No data."}

def get_logistics(city: str) -> dict:
    """Get travel logistics.
    Args:
        city: City name.
    Returns:
        Dict with flights, local transport, best season.
    """
    data = {"tokyo": {"flights": "3h from SFO", "local": "Shinkansen", "season": "Mar-May"},
            "paris": {"flights": "11h from NYC", "local": "Metro", "season": "Apr-Jun"},
            "bangalore": {"flights": "9h from Dubai", "local": "Ola/Uber", "season": "Oct-Feb"}}
    return {"city": city, **data[city.lower()]} if city.lower() in data else {"error": f"No data."}

print("Tools ready")

In [ ]:
highlights_agent = Agent(name="highlights_specialist", model="gemini-2.0-flash",
    description="Researches tourist highlights. Use for 'what to see' queries.",
    instruction="Call get_highlights() and summarize top 3 attractions clearly.",
    tools=[get_highlights])

logistics_agent = Agent(name="logistics_specialist", model="gemini-2.0-flash",
    description="Researches travel logistics: flights, transport, season. Use for practical info.",
    instruction="Call get_logistics() and summarize flights, transport, and best season.",
    tools=[get_logistics])

orchestrator = Agent(name="trip_orchestrator", model="gemini-2.0-flash",
    description="Orchestrates trip package creation.",
    instruction="""Orchestrate a trip package:
1. Delegate to highlights_specialist for attractions.
2. Delegate to logistics_specialist for travel info.
3. Synthesize into a ## Trip Package with ### Highlights, ### Logistics, ### 3-Day Itinerary, ### Summary.""",
    sub_agents=[highlights_agent, logistics_agent])

print(f"Orchestrator: {orchestrator.name} | Sub-agents: {[a.name for a in orchestrator.sub_agents]}")

In [ ]:
async def run(city: str) -> str:
    ss = InMemorySessionService()
    session = await ss.create_session(app_name="trip_orchestrator", user_id="u1")
    runner = InMemoryRunner(agent=orchestrator, session_service=ss)
    final = ""
    async for event in runner.run_async(user_id=session.user_id, session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=f"Create trip package for {city}.")])):
        if hasattr(event, "tool_call") and event.tool_call:
            print(f"  [tool] {event.tool_call.name}")
        elif event.is_final_response() and event.content:
            for p in event.content.parts:
                if p.text: final += p.text
    return final

package = await run("Tokyo")
print("\n--- Trip Package ---")
print(package)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `sub_agents=` on orchestrator | ADK native delegation — no explicit routing code |
| `description=` guides transfer | Orchestrator reads descriptions to pick sub-agents |
| Orchestrator synthesizes output | Sub-agents return data; orchestrator writes the package |

### Orchestrator vs Hierarchical in ADK
| | Hierarchical | Orchestrator-Subagent |
|---|---|---|
| Levels | 3+ (manager → leads → workers) | 2 (orchestrator → specialists) |
| Manager role | Delegates to leads | Directly uses specialists |
| Structure | Fixed team | Dynamic task assignment |

### Next: [05-Pipeline →](../../05-Pipeline/LangChain/pipeline.ipynb)